# Chapter 2: Geometric Memory Systems as Ground Truth

**Monograph:** *How to Test and Validate Agentic AI Systems*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Understand how GMS embeds knowledge on the unit hypersphere $S^{m-1}$.
2. Compute geodesic distance, tension energy and holonomy defect on a trained store.
3. Retrieve exact numeric values via the ENM register with SHA-256 integrity.
4. Use link prediction and structural queries to explore the knowledge graph.
5. Grasp why these six geometric primitives are sufficient for agent verification.

## Prerequisites

- Chapter 1 notebook completed (familiarity with MockHarness and the running example).
- No API keys required. All exercises use the bundled `model_risk_assessment.md`.

---
## Preamble

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the GMS expert store (shared with Chapter 4 onward)
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()

stats = store.stats()
print(f"Entities:    {stats['entities']}")
print(f"Relations:   {stats['relations']}")
print(f"Triples:     {stats['triples']}")
print(f"ENM entries: {stats['enm_entries']}")

---
## 2.1 The Knowledge Graph on the Hypersphere

A GMS expert store embeds all knowledge from the source document onto the unit hypersphere $S^{m-1}$. Every entity -- models, metrics, risk ratings, statuses -- becomes a point on this sphere. Every relation -- `has_status`, `has_test_auc`, `has_downstream_model` -- becomes a **relation operator**: a learned rotation matrix $R_r$ that transports one entity's embedding toward another's.

The store we just trained contains:
- **508 entities** (model IDs, metric names, status labels, dollar amounts, etc.)
- **66 relation types** extracted from the document structure
- **807 deduplicated knowledge triples** of the form (head, relation, tail)
- **261 exact numeric values** stored with SHA-256 integrity hashes

Let's explore the structure.

In [3]:
# Inspect the knowledge triples for MDL-001
triples_001 = store.query_triples(head="MDL-001")
print(f"MDL-001 has {len(triples_001)} triples. First 15:")
for h, r, t in triples_001[:15]:
    print(f"  ({h}, {r}, {t})")

MDL-001 has 82 triples. First 15:
  (MDL-001, has_tier, 1.0)
  (MDL-001, has_model_name, CreditScore-XGB)
  (MDL-001, has_domain, Credit Decisioning)
  (MDL-001, has_risk_rating, High)
  (MDL-001, has_status, Degraded)
  (MDL-001, in_section, model_inventory_summary)
  (MDL-001, has_features, 142.0)
  (MDL-001, has_algorithm, XGBoost (depth=6))
  (MDL-001, has_training_samples, 2,450,000)
  (MDL-001, has_training_date, 2025-09-15)
  (MDL-001, has_retrain_freq, Quarterly)
  (MDL-001, has_train_auc, 0.8934)
  (MDL-001, has_test_auc, 0.8412)
  (MDL-001, has_train-test_gap, 0.0522)
  (MDL-001, has_gini, 0.6824)


In [4]:
# List all 66 relation types
relations = sorted(store.adapter.relation_to_idx.keys())
print(f"All {len(relations)} relation types:")
for r in relations:
    print(f"  {r}")

All 66 relation types:
  fails
  has_%_complete
  has_acceptable_threshold
  has_accuracy_score
  has_action
  has_adverse_impact_ratio
  has_algorithm
  has_annual_loss_exposure
  has_annual_revenue_impact
  has_approval_rate
  has_brier_score
  has_capital_impact
  has_composite_risk_score
  has_current_psi
  has_data_quality_score
  has_delta
  has_delta_auc
  has_dependency_type
  has_description
  has_detail
  has_domain
  has_downstream_model
  has_drift_status
  has_due_date
  has_fairness_score
  has_feature_category
  has_feature_set
  has_features
  has_finding_id
  has_gini
  has_impact_if_upstream_fails
  has_ks_statistic
  has_missing_rate_q1_2026
  has_missing_rate_q4_2025
  has_model_id
  has_model_name
  has_noise_1%
  has_noise_10%
  has_noise_20%
  has_noise_5%
  has_owner
  has_protected_group
  has_reference_rate
  has_regulatory_penalty_risk
  has_remediation_deadline
  has_retrain_freq
  has_risk_rating
  has_risk_tier
  has_robustness_rating
  has_robustness_scor

---
## 2.2 Geometric Embeddings

Before we can measure distances and transformations on the sphere, we need to understand how entities get there. Every entity receives a **geometric embedding** -- a learned vector projected onto the unit hypersphere $S^{m-1}$ (with $m = 128$) via an inner-product-preserving projection. The constraint-aware optimizer ensures that every embedding has unit norm, so all entities live on the same sphere.

GMS maintains two complementary embedding views:
- **Semantic view:** Captures what an entity *means*. Used for relation transport and triple scoring.
- **Logical view:** Captures logical relationships. Used for tension energy and contradiction detection.

Both views are 128-dimensional unit vectors on $S^{m-1}$. Let's verify the manifold constraint.

In [5]:
model = store.model
adapter = store.adapter

# Architecture summary
print("=== Model Architecture ===")
print(f"  Entities:    {model.num_entities}")
print(f"  Relations:   {model.num_relations}")
print(f"  Embedding dim (m): {model.cfg.m}")

# Extract MDL-001's embedding and verify unit norm
idx_001 = adapter.entity_to_idx["MDL-001"]
idx_t = torch.tensor([idx_001], device=DEVICE)

with torch.no_grad():
    emb_001 = model.dual_emb.project_v(idx_t)

print("\nMDL-001 embedding:")
print(f"  shape: {emb_001.shape}")
print(f"  ||v|| = {emb_001.norm().item():.6f}")

# Verify all 508 embeddings lie on the unit sphere
with torch.no_grad():
    all_idx = torch.arange(adapter.num_entities, device=DEVICE)
    all_emb = model.dual_emb.project_v(all_idx)
    max_dev = (all_emb.norm(dim=1) - 1.0).abs().max()

print(f"\nAll {adapter.num_entities} embeddings on S^{{127}}:")
print(f"  max |norm - 1| = {max_dev:.2e}")

=== Model Architecture ===
  Entities:    508
  Relations:   66
  Embedding dim (m): 128

MDL-001 embedding:
  shape: torch.Size([1, 128])
  ||v|| = 1.000000

All 508 embeddings on S^{127}:
  max |norm - 1| = 1.19e-07


In [6]:
# The embedding is a 128-dimensional unit vector on the hypersphere.
# Let's inspect the first 8 components and verify the norm.
print("MDL-001 on the hypersphere:")
print(f"  dim = {emb_001.shape[1]}")
print(f"  ||v|| = {emb_001.norm().item():.6f}")
print(f"  v[:8] = {[f'{x:.4f}' for x in emb_001[0,:8].cpu().tolist()]}")
print("\nAll vectors have unit norm -- they live on S^{127}.")

MDL-001 on the hypersphere:
  dim = 128
  ||v|| = 1.000000
  v[:8] = ['-0.1151', '0.0135', '0.0812', '-0.0293', '-0.2632', '-0.0199', '0.0573', '-0.0143']

All vectors have unit norm -- they live on S^{127}.


---
## 2.3 Relation Operators: Relations as Geometric Transformations

In GMS, a relation $r$ is represented by a **relation operator** -- a learned rotation matrix $R_r \in SO(m)$ that maps one entity's embedding toward another's on the hypersphere. The operator is constrained to be a proper rotation ($R_r R_r^T = I$, $\det R_r = 1$), ensuring that it preserves the geometric structure of the sphere throughout training.

Given a triple $(h, r, t)$, the GMS scores it by applying the relation operator to $h$'s embedding and measuring how close the result lands to $t$'s embedding:

$$d(h, r, t) = \arccos\bigl(\langle R_r \tilde{v}_h, \, \tilde{v}_t \rangle\bigr)$$

A low geodesic distance means the triple is **plausible** according to the trained model.

In [7]:
# Verify relation operator properties: R R^T = I, det(R) = 1
print("=== Relation Operator Verification ===")
rels = ["has_status", "has_domain", "has_risk_rating",
        "has_test_auc", "has_downstream_model"]

for rname in rels:
    r_idx = adapter.relation_to_idx[rname]
    with torch.no_grad():
        R_r = model.relation_rotors[r_idx].get_rotor()
        RRt = R_r @ R_r.T
        I = torch.eye(R_r.shape[0], device=R_r.device)
        orth_err = (RRt - I).abs().max().item()
        det = torch.linalg.det(R_r).item()
    print(f"  {rname:30s}  |RR^T - I|_max = {orth_err:.8f}"
          f"  det = {det:.4f}")

=== Relation Operator Verification ===
  has_status                      |RR^T - I|_max = 0.00000370  det = 0.9999
  has_domain                      |RR^T - I|_max = 0.00000328  det = 0.9999
  has_risk_rating                 |RR^T - I|_max = 0.00000328  det = 0.9999
  has_test_auc                    |RR^T - I|_max = 0.00000340  det = 0.9999
  has_downstream_model            |RR^T - I|_max = 0.00000346  det = 0.9999


Every relation operator satisfies $RR^T = I$ to within floating-point precision ($< 10^{-5}$) and has determinant $\approx 1.0$. These are proper rotations on $S^{127}$.

---
## Lab 2.1 -- Primitive 1: `score_triple` (Fact Plausibility)

`score_triple(h, r, t)` returns the geodesic distance between $R_r \tilde{v}_h$ and $\tilde{v}_t$. A **low score** means the triple is plausible; a **high score** means it is unlikely.

This is the primary primitive for detecting **factual hallucination** (FM-1 from Chapter 1). Recall from Lab 1.1 that the mock agent returned 0.8934 (train AUC) instead of the correct 0.8412 (test AUC) for MDL-001. Both values are valid numbers from the document, so string matching cannot distinguish which is correct. With `score_triple` we can score both claims geometrically: the correct triple `(MDL-001, has_test_auc, 0.8412)` should yield a lower geodesic than the incorrect `(MDL-001, has_test_auc, 0.8934)`.

We now demonstrate this separation systematically, scoring ten known-true triples and nine fabricated triples with the wrong tail entity.

In [8]:
# Score known-true triples from the document
print("=== Known-True Triples (from the document) ===")
true_triples = [
    ("MDL-001", "has_status", "Degraded"),
    ("MDL-001", "has_domain", "Credit Decisioning"),
    ("MDL-001", "has_risk_rating", "High"),
    ("MDL-002", "has_domain", "Fraud Detection"),
    ("MDL-002", "has_risk_rating", "High"),
    ("MDL-003", "has_status", "Degraded"),
    ("MDL-004", "has_domain", "Market Risk VaR"),
    ("MDL-005", "has_status", "Degraded"),
    ("MDL-007", "has_status", "Stable"),
    ("MDL-010", "has_risk_rating", "Low"),
]

true_scores = []
for h, r, t in true_triples:
    score = store.score_triple(h, r, t)
    if score is not None:
        true_scores.append(score)
        print(f"  ({h}, {r}, {t}):  d = {score:.4f}")

import statistics
print(f"\n  Mean: {statistics.mean(true_scores):.4f}  "
      f"Std: {statistics.stdev(true_scores):.4f}")

=== Known-True Triples (from the document) ===
  (MDL-001, has_status, Degraded):  d = 1.2212
  (MDL-001, has_domain, Credit Decisioning):  d = 1.1370
  (MDL-001, has_risk_rating, High):  d = 1.3323
  (MDL-002, has_domain, Fraud Detection):  d = 1.3044
  (MDL-002, has_risk_rating, High):  d = 1.1832


  (MDL-003, has_status, Degraded):  d = 1.1742


  (MDL-004, has_domain, Market Risk VaR):  d = 1.2381


  (MDL-005, has_status, Degraded):  d = 1.1876


  (MDL-007, has_status, Stable):  d = 0.8819


  (MDL-010, has_risk_rating, Low):  d = 0.8524

  Mean: 1.1512  Std: 0.1611


In [9]:
# Score fabricated triples (wrong tail entities)
print("=== Fabricated Triples (wrong entity for each relation) ===")
fab_triples = [
    ("MDL-001", "has_domain", "Fraud Detection"),     # wrong domain
    ("MDL-001", "has_risk_rating", "Low"),             # wrong risk
    ("MDL-002", "has_domain", "Credit Decisioning"),   # wrong domain
    ("MDL-002", "has_risk_rating", "Low"),             # wrong risk
    ("MDL-003", "has_status", "Stable"),               # wrong status
    ("MDL-004", "has_domain", "Customer Analytics"),   # wrong domain
    ("MDL-005", "has_status", "Stable"),               # wrong status
    ("MDL-007", "has_status", "Degraded"),             # wrong status
    ("MDL-010", "has_risk_rating", "High"),            # wrong risk
]

fab_scores = []
for h, r, t in fab_triples:
    score = store.score_triple(h, r, t)
    if score is not None:
        fab_scores.append(score)
        print(f"  ({h}, {r}, {t}):  d = {score:.4f}")

print(f"\n  Mean: {statistics.mean(fab_scores):.4f}  "
      f"Std: {statistics.stdev(fab_scores):.4f}")

print("\n=== Comparison ===")
print(f"  True triples mean:       {statistics.mean(true_scores):.4f}")
print(f"  Fabricated triples mean: {statistics.mean(fab_scores):.4f}")
print(f"  Separation:              {statistics.mean(fab_scores) - statistics.mean(true_scores):+.4f}")
print("\n  Fabricated triples score higher (further from ground truth).")

=== Fabricated Triples (wrong entity for each relation) ===
  (MDL-001, has_domain, Fraud Detection):  d = 1.6897
  (MDL-001, has_risk_rating, Low):  d = 1.6229
  (MDL-002, has_domain, Credit Decisioning):  d = 1.5431
  (MDL-002, has_risk_rating, Low):  d = 1.5248
  (MDL-003, has_status, Stable):  d = 1.3969
  (MDL-004, has_domain, Customer Analytics):  d = 1.5389


  (MDL-005, has_status, Stable):  d = 1.3909


  (MDL-007, has_status, Degraded):  d = 1.4997


  (MDL-010, has_risk_rating, High):  d = 1.5054

  Mean: 1.5236  Std: 0.0953

=== Comparison ===
  True triples mean:       1.1512
  Fabricated triples mean: 1.5236
  Separation:              +0.3724

  Fabricated triples score higher (further from ground truth).


---
## Lab 2.2 -- Primitive 2: `tension_energy` (Contradiction Detection)

Tension energy operates in the **logical embedding space**. It measures how strongly two entities agree, are irrelevant to each other or contradict each other:

$$E(u_a, u_b) = \sqrt{2 - 2 \langle u_a, u_b \rangle} = 2 \sin\!\left(\frac{\theta}{2}\right)$$

The energy spectrum has three semantic bands:

| Energy Range | Meaning | Geometric Interpretation |
|---|---|---|
| $E \approx 0$ | Agreement | Vectors aligned ($\theta \approx 0$) |
| $E \approx \sqrt{2} \approx 1.414$ | Irrelevance | Vectors orthogonal ($\theta \approx \pi/2$) |
| $E \approx 2$ | Contradiction | Vectors antipodal ($\theta \approx \pi$) |

At the 500 training epochs used throughout this book, the spectrum separates cleanly into these three bands: agreement-class pairs (same-risk models) sit well below $\sqrt{2}$, unrelated pairs cluster near the irrelevance boundary, and true semantic opposites push toward $E \to 2$. This is the same training depth used for every store from Chapter 4 onward.

This primitive detects when an agent's answer **contradicts** the source document.

In [10]:
# Compute tension energy across different entity pairs
import math

print("=== Tension Energy: Same-Risk Model Pairs ===")
print("  (All four models below are rated High risk)")
for a, b in [("MDL-001", "MDL-002"), ("MDL-001", "MDL-003"),
             ("MDL-002", "MDL-003")]:
    t = store.tension_energy(a, b)
    print(f"  E({a}, {b}) = {t:.4f}")

print("\n=== Tension Energy: Cross-Risk Model Pairs ===")
for a, b in [("MDL-001", "MDL-005"), ("MDL-002", "MDL-010"),
             ("MDL-003", "MDL-007")]:
    t = store.tension_energy(a, b)
    print(f"  E({a}, {b}) = {t:.4f}")

print("\n=== Tension Energy: Semantic Opposites ===")
for a, b in [("High", "Low"), ("Degraded", "Stable"),
             ("Critical", "Low")]:
    t = store.tension_energy(a, b)
    if t is not None:
        print(f"  E({a:10s}, {b:10s}) = {t:.4f}")
    else:
        print(f"  E({a:10s}, {b:10s}) = None")

print(f"\n  Reference: sqrt(2) = {math.sqrt(2):.4f} (irrelevance boundary)")

=== Tension Energy: Same-Risk Model Pairs ===
  (All four models below are rated High risk)
  E(MDL-001, MDL-002) = 0.7078
  E(MDL-001, MDL-003) = 0.6225
  E(MDL-002, MDL-003) = 0.6572

=== Tension Energy: Cross-Risk Model Pairs ===
  E(MDL-001, MDL-005) = 0.6026
  E(MDL-002, MDL-010) = 0.6978
  E(MDL-003, MDL-007) = 0.5849

=== Tension Energy: Semantic Opposites ===
  E(High      , Low       ) = 1.4129
  E(Degraded  , Stable    ) = 1.3015
  E(Critical  , Low       ) = 1.4827

  Reference: sqrt(2) = 1.4142 (irrelevance boundary)


---
## Lab 2.3 -- Primitive 3: `check_holonomy` (Path Consistency)

When we compose two relation operators -- for example, `has_train_auc` followed by `has_test_auc` -- we get a path operator $R_{\text{path}} = R_2 R_1$. The **holonomy defect** compares this composed path to a single direct relation operator:

$$H = \| R_{\text{path}} \, R_{\text{direct}}^{-1} - I \|_F$$

If $H \approx 0$, the two paths are geometrically consistent -- like walking two sides of a triangle and arriving at the same point as the shortcut. If $H$ is large, the paths disagree, indicating an inconsistency in the agent's multi-hop reasoning.

**Interpreting the magnitude:** A random pair of $128 \times 128$ rotation matrices would produce a Frobenius defect of $\|R_1 R_2^{-1} - I\|_F \approx \sqrt{2m} \approx 16.0$; the identity (perfect consistency) gives exactly 0. Our trained defects of $\approx 2.5$ sit well below random, indicating meaningful geometric structure.

This is the geometric analog of **logical consistency** and detects **FM-4: Multi-Hop Reasoning Failure** from Chapter 1.

In [11]:
# Check holonomy defects for various relation paths
print("=== Holonomy Defect ===")

holonomy_cases = [
    (["has_domain", "has_risk_rating"], "has_status",
     "domain + risk_rating vs status"),
    (["has_train_auc", "has_test_auc"], "has_accuracy_score",
     "train_auc + test_auc vs accuracy_score"),
    (["has_noise_1%", "has_noise_5%"], "has_robustness_rating",
     "noise_1% + noise_5% vs robustness_rating"),
]

for path, direct, description in holonomy_cases:
    h = store.check_holonomy(path, direct)
    if h is not None:
        print(f"  {description}")
        print(f"    path={path} -> direct={direct}")
        print(f"    holonomy defect = {h:.4f}")
        print()

=== Holonomy Defect ===
  domain + risk_rating vs status
    path=['has_domain', 'has_risk_rating'] -> direct=has_status
    holonomy defect = 2.6750

  train_auc + test_auc vs accuracy_score
    path=['has_train_auc', 'has_test_auc'] -> direct=has_accuracy_score
    holonomy defect = 2.6682

  noise_1% + noise_5% vs robustness_rating
    path=['has_noise_1%', 'has_noise_5%'] -> direct=has_robustness_rating
    holonomy defect = 2.6826



---
## Lab 2.4 -- Primitive 4: `lookup_enm` (Exact Numerical Memory)

Some facts are inherently numeric: "the test AUC of MDL-001 is 0.8412". Embedding a number as a point on a sphere would lose precision -- the embedding might reconstruct 0.84 or 0.842 but never exactly 0.8412. For these facts GMS uses a separate **Exact Numerical Memory (ENM)** register.

ENM stores each numeric value as:
- An IEEE 754 float64 byte representation
- A **SHA-256 hash** computed over those bytes
- A key of the form `(section_type, entity_path)`

On retrieval, the hash is recomputed and verified. If a single bit has changed, the integrity check fails. This is how GMSH detects **FM-6: State Corruption** -- if a stored value is tampered with, the SHA-256 hash will not match.

In [12]:
# Look up exact numeric values from the ENM register
print("=== ENM Lookup: Performance Metrics ===")
enm_lookups = [
    ("performance_metrics", "MDL-001/Train AUC"),
    ("performance_metrics", "MDL-001/Test AUC"),
    ("performance_metrics", "MDL-001/Gini"),
    ("performance_metrics", "MDL-001/KS Statistic"),
    ("performance_metrics", "MDL-001/Brier Score"),
    ("performance_metrics", "MDL-002/Test AUC"),
    ("performance_metrics", "MDL-005/Test AUC"),
]

for cat, eid in enm_lookups:
    val = store.lookup_enm(cat, eid)
    if val is not None:
        print(f"  lookup_enm({cat!r}, {eid!r}) = {val}")
    else:
        print(f"  lookup_enm({cat!r}, {eid!r}) = None")

# Composite risk scores live under a different section
print("\n=== ENM Lookup: Composite Risk Scores ===")
risk_lookups = [
    ("risk_findings_and_remediation",
     "MDL-001/45/Critical/Composite Risk Score"),
    ("risk_findings_and_remediation",
     "MDL-005/N/A/Critical/Composite Risk Score"),
    ("risk_findings_and_remediation",
     "MDL-010/N/A/Low/Composite Risk Score"),
]
for cat, eid in risk_lookups:
    val = store.lookup_enm(cat, eid)
    if val is not None:
        print(f"  {eid}: {val}")
    else:
        print(f"  {eid}: None")

=== ENM Lookup: Performance Metrics ===
  lookup_enm('performance_metrics', 'MDL-001/Train AUC') = 0.8934
  lookup_enm('performance_metrics', 'MDL-001/Test AUC') = 0.8412
  lookup_enm('performance_metrics', 'MDL-001/Gini') = 0.6824
  lookup_enm('performance_metrics', 'MDL-001/KS Statistic') = 0.4891
  lookup_enm('performance_metrics', 'MDL-001/Brier Score') = 0.1523
  lookup_enm('performance_metrics', 'MDL-002/Test AUC') = 0.9234
  lookup_enm('performance_metrics', 'MDL-005/Test AUC') = 0.7234

=== ENM Lookup: Composite Risk Scores ===
  MDL-001/45/Critical/Composite Risk Score: 56.75
  MDL-005/N/A/Critical/Composite Risk Score: 41.33
  MDL-010/N/A/Low/Composite Risk Score: 90.67


In [ ]:
# Verify SHA-256 integrity of all 261 ENM entries
print("=== ENM Integrity Verification ===")

all_keys = list(store.enm._store.keys())
n_verified = 0
n_failed = 0
for k in all_keys:
    ok = store.enm.verify_integrity(k)
    if ok:
        n_verified += 1
    else:
        n_failed += 1

print(f"  {n_verified}/{len(all_keys)} entries passed SHA-256 verification")
print(f"  {n_failed}/{len(all_keys)} entries failed")

# Show the hash for one entry
sample_key = [k for k in all_keys
              if k.id == "MDL-001/Test AUC"][0]
entry = store.enm._store[sample_key]
print("\n  Example: MDL-001/Test AUC")
print(f"    Value:     {float(entry.value):.4f}")
print(f"    SHA-256:   {entry.hash.hex()[:32]}...")
print(f"    Integrity: {store.enm.verify_integrity(sample_key)}")

All 261 entries pass integrity verification. The ENM provides **lossless** numeric recall -- unlike the embeddings, which trade precision for generalization, ENM stores every digit exactly as it appeared in the source document.

---
## Lab 2.5 -- Primitive 5: `link_predict` (Entity Prediction)

`link_predict(h, r, top_k)` answers the question: "given head entity $h$ and relation $r$, which tail entities are most plausible?" It computes $d(h, r, t)$ for every entity $t$ in the vocabulary and returns the top-$k$ with the lowest geodesic distance.

At the 500 training epochs used here, the correct tail entities (e.g., "High", "Credit Decisioning") consistently appear in the top ranks, and `link_predict` is the primitive the GMSH generators use to build *distractor sets* -- plausible but incorrect candidate answers for multiple-choice question generation. Oracles use it to check whether an agent's answer falls within the geometrically plausible neighborhood. For exact entity lookup, `query_triples` and `lookup_enm` remain the appropriate primitives.

In [14]:
# Link prediction: (MDL-001, has_risk_rating, ?)
print("=== link_predict: (MDL-001, has_risk_rating, ?) ===")
print("  Expected: High (from document)")
preds = store.link_predict("MDL-001", "has_risk_rating", top_k=5)
for entity, score in preds:
    print(f"    {entity:35s}  score = {score:.4f}")

print("\n=== link_predict: (MDL-001, has_domain, ?) ===")
print("  Expected: Credit Decisioning")
preds = store.link_predict("MDL-001", "has_domain", top_k=5)
for entity, score in preds:
    print(f"    {entity:35s}  score = {score:.4f}")

print("\n=== link_predict: (MDL-001, has_downstream_model, ?) ===")
print("  Expected: MDL-008 (from dependency table)")
preds = store.link_predict("MDL-001", "has_downstream_model", top_k=5)
for entity, score in preds:
    print(f"    {entity:35s}  score = {score:.4f}")

=== link_predict: (MDL-001, has_risk_rating, ?) ===
  Expected: High (from document)
    High                                 score = 1.3323
    Medium                               score = 1.4906
    Low                                  score = 1.6229

=== link_predict: (MDL-001, has_domain, ?) ===
  Expected: Credit Decisioning
    Credit Decisioning                   score = 1.1370
    Probability of Default               score = 1.4660
    Customer Analytics                   score = 1.5147
    Interest Rate Pricing                score = 1.5701
    Collateral Valuation                 score = 1.5766

=== link_predict: (MDL-001, has_downstream_model, ?) ===
  Expected: MDL-008 (from dependency table)
    MDL-008                              score = 1.3689
    MDL-003                              score = 1.4578
    MDL-005                              score = 1.4640
    MDL-010                              score = 1.6042


---
## Lab 2.6 -- Primitive 6: `query_triples` (Structural Query)

`query_triples` performs pattern matching on the document graph -- the raw triples extracted during ingestion. Unlike the other primitives, this is **not** a learned geometric operation. It simply filters the stored triples by head, relation and/or tail.

This is used for structural verification: does the document actually contain a triple that matches the agent's claim? If the agent says "MDL-002 has domain Fraud Detection", we can check whether the triple `(MDL-002, has_domain, Fraud Detection)` exists in the graph.

In [15]:
# Query by head: all facts about MDL-002
print("=== query_triples(head='MDL-002') — first 10 ===")
for h, r, t in store.query_triples(head="MDL-002")[:10]:
    print(f"  ({h}, {r}, {t})")

# Query by relation: all risk ratings
print("\n=== query_triples(relation='has_risk_rating') ===")
for h, r, t in store.query_triples(relation="has_risk_rating"):
    print(f"  ({h}, {r}, {t})")

# Query by tail: who has domain "Fraud Detection"?
print("\n=== query_triples(tail='Fraud Detection') ===")
for h, r, t in store.query_triples(tail="Fraud Detection"):
    print(f"  ({h}, {r}, {t})")

=== query_triples(head='MDL-002') — first 10 ===
  (MDL-002, has_tier, 1.0)
  (MDL-002, has_model_name, FraudNet-LSTM)
  (MDL-002, has_domain, Fraud Detection)
  (MDL-002, has_risk_rating, High)
  (MDL-002, has_status, Watch)
  (MDL-002, in_section, model_inventory_summary)
  (MDL-002, has_features, 87.0)
  (MDL-002, has_algorithm, LSTM (3-layer))
  (MDL-002, has_training_samples, 8,900,000)
  (MDL-002, has_training_date, 2025-11-01)

=== query_triples(relation='has_risk_rating') ===
  (MDL-001, has_risk_rating, High)
  (MDL-002, has_risk_rating, High)
  (MDL-003, has_risk_rating, High)
  (MDL-004, has_risk_rating, High)
  (MDL-005, has_risk_rating, Medium)
  (MDL-006, has_risk_rating, Medium)
  (MDL-007, has_risk_rating, Medium)
  (MDL-008, has_risk_rating, Medium)
  (MDL-009, has_risk_rating, Medium)
  (MDL-010, has_risk_rating, Low)
  (MDL-011, has_risk_rating, Low)
  (MDL-012, has_risk_rating, Low)

=== query_triples(tail='Fraud Detection') ===
  (MDL-002, has_domain, Fraud Detecti

---
## 2.4 Why Six Primitives Are Sufficient

Each of the six failure modes from Chapter 1 maps to exactly one geometric primitive:

| Failure Mode | Primitive | What It Checks |
|---|---|---|
| FM-1: Hallucination | `score_triple` | Geodesic distance to ground truth triple |
| FM-2: Tool misuse | `query_triples` | Wrong relation type in tool trace |
| FM-3: Policy violation | `query_triples` | Forbidden action predicate in trace |
| FM-4: Multi-hop failure | `check_holonomy` | Path consistency across composed operators |
| FM-5: Miscalibration | Phase encoding | Phase--accuracy divergence (Chapter 17) |
| FM-6: State corruption | `lookup_enm` | SHA-256 hash mismatch on numeric recall |

Together these six primitives give us a **complete geometric verification vocabulary**. Any claim an agent makes can be decomposed into one or more primitive checks. For example, suppose an agent answers: "MDL-001 has test AUC 0.84 and is rated Low risk." This single sentence decomposes into three checks:

1. `lookup_enm("performance_metrics", "MDL-001/Test AUC")` -- is 0.84 the exact stored value? (It is not; the ENM stores 0.8412.)
2. `score_triple("MDL-001", "has_risk_rating", "Low")` -- is "Low" plausible? (High geodesic: the document says High.)
3. `tension_energy("Low", "High")` -- does the claimed "Low" contradict the known "High"? (Yes: elevated tension.)

The typed verification pipeline in Part III formalizes this decomposition.

---
## Exercises

### Exercise 2.1

Score all 12 models on the triple `(MDL-XXX, has_risk_rating, High)`. Which models score lowest (most plausible as High risk)? Do the results match the document's actual risk ratings?

### Exercise 2.2

Compute `tension_energy` for all pairs among MDL-001 through MDL-006 (15 pairs). Which pair has the highest tension? Can you explain why from the document content?

### Exercise 2.3

Use `lookup_enm` to retrieve the test AUC for all 12 models. Compute the train-test gap by also looking up the train AUC. Verify that your computed gaps match the `has_train-test_gap` values stored as triples.

In [16]:
# === Exercise 2.1: Your code here ===


In [17]:
# === Exercise 2.2: Your code here ===


In [18]:
# === Exercise 2.3: Your code here ===


---
## Summary

In this chapter we explored the six geometric primitives that form the verification vocabulary of GMSH:

1. **`score_triple`** measures geodesic distance on $S^{m-1}$ -- the core signal for fact plausibility. True triples score lower than fabricated ones.

2. **`tension_energy`** uses the logical embedding space to detect contradictions. The energy spectrum ranges from 0 (agreement) through $\sqrt{2}$ (irrelevance) to 2 (contradiction).

3. **`check_holonomy`** compares composed path operators to direct operators, detecting inconsistencies in multi-hop reasoning chains.

4. **`lookup_enm`** provides lossless numeric recall with SHA-256 integrity -- no embedding approximation, no rounding error.

5. **`link_predict`** performs beam search on the manifold to find the most plausible tail entities for a given (head, relation) pair.

6. **`query_triples`** provides exact structural matching on the document graph for direct fact verification.

We also verified the mathematical guarantees that make these primitives reliable:
- Entity embeddings live on the unit sphere ($\|\tilde{v}\| = 1$, deviation $< 10^{-7}$)
- Relation operators are proper rotations ($RR^T = I$, $\det R = 1$)
- ENM entries pass SHA-256 integrity verification (261/261)

In the next chapter we learn how Design of Experiments turns these primitives into systematic, statistically rigorous test batteries.